In [ ]:
import os
import zipfile
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.utils.class_weight import compute_class_weight

# CONFIG
ZIP_PATH = "/content/BRAHMI.zip"
EXTRACT_DIR = "/content/BRAHMI_dataset"
MODEL_SAVE_PATH = "/content/brahmi_model.h5"
IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 100
PATIENCE_ES = 7
PATIENCE_LR = 3

# 1. Extract the dataset
def extract_dataset(zip_path, extract_to):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
    print(f"Extracted dataset to: {extract_to}")
    print("Sample folders:", os.listdir(extract_to))

# 2. Data generators
def create_data_generators(data_dir):
    datagen = ImageDataGenerator(
        preprocessing_function=preprocess_input,
        rotation_range=20,
        width_shift_range=0.1,
        height_shift_range=0.1,
        shear_range=0.1,
        zoom_range=0.2,
        horizontal_flip=True,
        validation_split=0.15
    )

    train_gen = datagen.flow_from_directory(
        data_dir,
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        subset='training',
        shuffle=True
    )

    val_gen = datagen.flow_from_directory(
        data_dir,
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        subset='validation',
        shuffle=False
    )

    return train_gen, val_gen

# 3. Build model
def build_model(num_classes):
    base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))

    for layer in base_model.layers[:-10]:
        layer.trainable = False

    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.5)(x)
    x = Dense(1024, activation='relu')(x)
    x = Dropout(0.3)(x)
    predictions = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=base_model.input, outputs=predictions)

    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

    return model

# 4. Class weights
def get_class_weights(generator):
    y = generator.classes
    class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y), y=y)
    return dict(enumerate(class_weights))

# 5. Callbacks
def get_callbacks():
    os.makedirs(os.path.dirname(MODEL_SAVE_PATH), exist_ok=True)
    return [
        EarlyStopping(monitor='val_loss', patience=PATIENCE_ES, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', patience=PATIENCE_LR, factor=0.2, verbose=1),
        ModelCheckpoint(MODEL_SAVE_PATH, monitor='val_accuracy', save_best_only=True, verbose=1)
    ]

# 6. Train pipeline
def train():
    extract_dataset(ZIP_PATH, EXTRACT_DIR)
    train_gen, val_gen = create_data_generators(EXTRACT_DIR)
    num_classes = len(train_gen.class_indices)
    model = build_model(num_classes)
    class_weights = get_class_weights(train_gen)

    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=EPOCHS,
        class_weight=class_weights,
        callbacks=get_callbacks()
    )

    print("Training completed.")
    return model, history

# 7. Run it
if __name__ == "__main__":
    model, history = train()
